In [16]:
import pandas as pd
import sqlite3

## 1. Connection

In [17]:
db = sqlite3.connect("../data/checking-logs.sqlite")

## 2. Schema

In [18]:
pd.io.sql.read_sql("PRAGMA table_info(test);", db)


,cid,name,type,notnull,dflt_value,pk
0,0,uid,TEXT,0,None,0
1,1,labname,TEXT,0,None,0
2,2,first_commit_ts,TIMESTAMP,0,None,0
3,3,first_view_ts,TIMESTAMP,0,None,0


## 3. First 10

In [19]:
pd.io.sql.read_sql("SELECT* FROM test LIMIT 10", db)

,uid,labname,first_commit_ts,first_view_ts
0,user_1,laba04,2020-04-26 17:06:18,2020-04-26 21:53:59
1,user_1,laba04s,2020-04-26 17:12:11,2020-04-26 21:53:59
2,user_1,laba05,2020-05-02 19:15:18,2020-04-26 21:53:59
3,user_1,laba06,2020-05-17 16:26:35,2020-04-26 21:53:59
4,user_1,laba06s,2020-05-20 12:23:37,2020-04-26 21:53:59
5,user_1,project1,2020-05-14 20:56:08,2020-04-26 21:53:59
6,user_10,laba04,2020-04-25 08:24:52,2020-04-18 12:19:50
7,user_10,laba04s,2020-04-25 08:37:54,2020-04-18 12:19:50
8,user_10,laba05,2020-05-01 19:27:26,2020-04-18 12:19:50
9,user_10,laba06,2020-05-19 11:39:28,2020-04-18 12:19:50


## 4. Finding the minimum value of the delta between the first commit and the deadline

- немного отображений для себя

In [20]:
pd.io.sql.read_sql("PRAGMA table_info(deadlines);", db)


,cid,name,type,notnull,dflt_value,pk
0,0,index,INTEGER,0,None,0
1,1,labs,TEXT,0,None,0
2,2,deadlines,INTEGER,0,None,0


In [21]:
pd.io.sql.read_sql("SELECT* FROM deadlines LIMIT 10", db)

,index,labs,deadlines
0,0,laba04,1587945599
1,1,laba04s,1587945599
2,2,laba05,1588550399
3,4,laba06,1590364799
4,5,laba06s,1590364799
5,3,project1,1589673599


- создаю один кусок запроса, который будет повторяться в будущих тасках

In [22]:
diff = """
WITH diffs AS
(SELECT
  l.uid,
  (strftime('%s', l.first_commit_ts) - r.deadlines) / 3600 AS diff
FROM test AS l
JOIN deadlines AS r
  ON r.labs = l.labname
WHERE l.labname != 'project1')
"""


- самая маленькая дельта между первым коммитом и дедлайном (в часах)
(по факту наоборот самая большая, потому что у нас числа отрицательные по тз)

In [23]:
df_min = pd.io.sql.read_sql(diff + "SELECT uid, MIN(diff) as min_diff from diffs", db)
df_min

,uid,min_diff
0,user_30,-202


## 5. Same with maximum 

In [24]:
df_max = pd.io.sql.read_sql(diff + "SELECT uid, MAX(diff) as max_diff from diffs", db)
df_max



,uid,max_diff
0,user_25,-2


## 6. Same with AVG
but without the *uid*

In [25]:
avg_query = diff + "SELECT AVG(diff) as avg_diff from diffs"
df_avg = pd.io.sql.read_sql(avg_query, db)
df_avg

,avg_diff
0,-89.125


## 7. Те кто смотрели новости имеют меньшую дельту

- по настоящему правильный ответ

In [26]:
query_7 = """
WITH 
uids AS 
(SELECT DISTINCT uid FROM test),
diffs AS
    (SELECT
        t.uid,
        (strftime('%s', t.first_commit_ts) - d.deadlines) / 3600 AS diff
    FROM test t
    JOIN deadlines d 
    ON d.labs = t.labname
    WHERE t.labname != 'project1'),
avg AS
    (SELECT uid, AVG(diff) AS avg_diff
    FROM diffs
    GROUP BY uid),
pv AS
    (SELECT uid, COUNT(*) AS pageviews
    FROM pageviews
    GROUP BY uid)
SELECT 
    uids.uid,
    avg.avg_diff,
    pv.pageviews
FROM uids
LEFT JOIN avg USING (uid)
LEFT JOIN pv USING (uid);
"""
views_diff = pd.io.sql.read_sql(query_7, db)
corr = views_diff[["avg_diff", "pageviews"]].corr()
corr
# views_diff


,avg_diff,pageviews
avg_diff,1.000000,-0.279736
pageviews,-0.279736,1.000000


- с неправильным учетом project1

In [27]:
query_7 = """
WITH 
uids AS 
(SELECT DISTINCT uid FROM test),
diffs AS
    (SELECT
        t.uid,
        (strftime('%s', t.first_commit_ts) - d.deadlines) / 3600 AS diff
    FROM test t
    JOIN deadlines d 
    ON d.labs = t.labname),
avg AS
    (SELECT uid, AVG(diff) AS avg_diff
    FROM diffs
    GROUP BY uid),
pv AS
    (SELECT uid, COUNT(*) AS pageviews
    FROM pageviews
    GROUP BY uid)
SELECT 
    uids.uid,
    avg.avg_diff,
    pv.pageviews
FROM uids
LEFT JOIN avg USING (uid)
LEFT JOIN pv USING (uid);
"""
views_diff = pd.io.sql.read_sql(query_7, db)
corr = views_diff[["avg_diff", "pageviews"]].corr()
corr
# views_diff


,avg_diff,pageviews
avg_diff,1.000000,-0.069302
pageviews,-0.069302,1.000000


## 8. Close connection

In [28]:
db.close()